# NewsSum V2: publish models and demo to Hugging Face

Publishes the three released NewsSum V2 checkpoints and a static browser demo:

| Repository | Content |
|---|---|
| `newssum-pegasus-large-bbc` | PEGASUS-large, the best fine-tuned model |
| `newssum-bart-base-bbc` | BART-base, plus a quantised ONNX copy for Transformers.js |
| `newssum-qwen2.5-1.5b-lora-bbc` | LoRA adapter for Qwen2.5-1.5B-Instruct |
| Space `newssum-demo` | Static page: TextRank and BART-base side by side, running in the browser |

### Before running
1. **Add Input → Your Work → Notebooks:** *NewsSum V2: BBC News Summarisation Benchmark* (its latest version, which holds `newssum/models/`).
2. **Add-ons → Secrets:** add `HF_TOKEN` with a Hugging Face token that has **write** access, and tick it for this notebook.
3. **Internet: On.** A CPU session is enough.

In [ ]:
!pip install -q --upgrade huggingface_hub onnxruntime
!pip install -q optimum-onnx || pip install -q "optimum[onnxruntime]"
!git clone -q --depth 1 https://github.com/MHKhan50/NewsSum-Bench.git /kaggle/working/repo

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
USER = api.whoami()["name"]
REPO = Path("/kaggle/working/repo")
MODELS = sorted({p.parent for p in Path("/kaggle/input").rglob("newssum/models/*/newssum_meta.json")})
MODELS = {p.name: p for p in MODELS}
print("Hugging Face user:", USER)
print("Models found:", list(MODELS))
assert {"pegasus-large", "bart-base", "qwen2.5-1.5b-lora"} <= set(MODELS), "Add the NewsSum V2 notebook output as input"

TARGETS = {
    "pegasus-large": ("newssum-pegasus-large-bbc", "pegasus-large.md"),
    "bart-base": ("newssum-bart-base-bbc", "bart-base.md"),
    "qwen2.5-1.5b-lora": ("newssum-qwen2.5-1.5b-lora-bbc", "qwen-lora.md"),
}

def card(name, repo_id):
    text = (REPO / "huggingface" / "cards" / name).read_text()
    return text.replace("{repo}", repo_id).replace("{user}", USER)

## 1. PEGASUS-large and the Qwen LoRA adapter

In [ ]:
for key in ["pegasus-large", "qwen2.5-1.5b-lora"]:
    repo_name, card_file = TARGETS[key]
    repo_id = f"{USER}/{repo_name}"
    api.create_repo(repo_id, exist_ok=True)
    api.upload_folder(repo_id=repo_id, folder_path=str(MODELS[key]), commit_message="Upload NewsSum V2 checkpoint",
                      ignore_patterns=["README.md"])
    api.upload_file(path_or_fileobj=card(card_file, repo_id).encode(), path_in_repo="README.md", repo_id=repo_id,
                    commit_message="Add model card")
    print("published https://huggingface.co/" + repo_id)

## 2. BART-base with a quantised ONNX copy for the browser

In [ ]:
import shutil, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

src = MODELS["bart-base"]
fp32 = Path("/kaggle/working/bart32")
shutil.rmtree(fp32, ignore_errors=True)
try:
    model = AutoModelForSeq2SeqLM.from_pretrained(src, dtype=torch.float32)
except TypeError:
    model = AutoModelForSeq2SeqLM.from_pretrained(src, torch_dtype=torch.float32)
model.save_pretrained(fp32)
AutoTokenizer.from_pretrained(src).save_pretrained(fp32)
for f in ["generation_config.json", "newssum_meta.json"]:
    if (src / f).exists() and not (fp32 / f).exists():
        shutil.copy(src / f, fp32 / f)

onnx_dir = Path("/kaggle/working/bart_onnx")
shutil.rmtree(onnx_dir, ignore_errors=True)
from optimum.exporters.onnx import main_export
main_export(model_name_or_path=str(fp32), output=onnx_dir, task="text2text-generation-with-past")
print(sorted(p.name for p in onnx_dir.glob("*.onnx")))

In [ ]:
from onnxruntime.quantization import QuantType, quantize_dynamic

out = fp32 / "onnx"
out.mkdir(exist_ok=True)
needed = ["encoder_model", "decoder_model_merged"]
for name in needed:
    srcf = onnx_dir / f"{name}.onnx"
    assert srcf.exists(), f"{srcf.name} missing; exported: {[p.name for p in onnx_dir.glob('*.onnx')]}"
    quantize_dynamic(str(srcf), str(out / f"{name}_quantized.onnx"), weight_type=QuantType.QInt8)
for p in sorted(out.iterdir()):
    print(p.name, f"{p.stat().st_size / 1e6:.0f} MB")

# Quick check: the quantised encoder/decoder produce a sensible summary with onnxruntime via optimum.
try:
    from optimum.onnxruntime import ORTModelForSeq2SeqLM
    tok = AutoTokenizer.from_pretrained(fp32)
    ort = ORTModelForSeq2SeqLM.from_pretrained(onnx_dir)
    text = (REPO / "huggingface" / "space" / "index.html").read_text().split("const EXAMPLE = `")[1].split("`;")[0]
    ids = tok(text, return_tensors="pt", truncation=True, max_length=1024)
    print(tok.decode(ort.generate(**ids, max_new_tokens=200, num_beams=1)[0], skip_special_tokens=True))
except Exception as e:
    print("ONNX check skipped:", e)

In [ ]:
repo_name, card_file = TARGETS["bart-base"]
BART = f"{USER}/{repo_name}"
api.create_repo(BART, exist_ok=True)
api.upload_folder(repo_id=BART, folder_path=str(src), commit_message="Upload NewsSum V2 checkpoint",
                  ignore_patterns=["README.md"])
api.upload_folder(repo_id=BART, folder_path=str(out), path_in_repo="onnx", commit_message="Add quantised ONNX for Transformers.js")
api.upload_file(path_or_fileobj=card(card_file, BART).encode(), path_in_repo="README.md", repo_id=BART,
                commit_message="Add model card")
print("published https://huggingface.co/" + BART)

## 3. Static demo Space

In [ ]:
SPACE = f"{USER}/newssum-demo"
api.create_repo(SPACE, repo_type="space", space_sdk="static", exist_ok=True)
space_dir = REPO / "huggingface" / "space"
html = (space_dir / "index.html").read_text().replace("{repo_bart}", BART)
api.upload_file(path_or_fileobj=html.encode(), path_in_repo="index.html", repo_id=SPACE, repo_type="space",
                commit_message="NewsSum V2 demo")
api.upload_file(path_or_fileobj=(space_dir / "README.md").read_bytes(), path_in_repo="README.md", repo_id=SPACE,
                repo_type="space", commit_message="Space card")
print("Demo: https://huggingface.co/spaces/" + SPACE)
print("Models:")
for key, (name, _) in TARGETS.items():
    print("  https://huggingface.co/" + f"{USER}/{name}")